<div style="background-color: #000000; padding: 10px; border-radius: 5px;">
<img src="images/logo.png" width="100">

In [34]:
# find the center of the ligand
import MDAnalysis as mda
original_structure = mda.Universe("protein_structures/chain_A.pdb")
ligand_mda = original_structure.select_atoms("resname TOP and resid 703")
# Get the center of the ligand as the "pocket center"
pocket_center = ligand_mda.center_of_geometry()## Fill in code for measuring the center of geometry of the ligand.
print(pocket_center)

[  1.21042854   5.8221429  -59.31728581]


In [35]:
from vina import Vina
v = Vina(sf_name="vina")

In [36]:
#! grep " A  86 " protein_structures/chain_A_cleanH.pdb

In [44]:
# !mk_prepare_receptor.py \
# -i protein_structures/chain_A_cleanH.pdb \
# -o protein_structures/chain_A_clean \
# -p

In [38]:
from vina import Vina
v = Vina(sf_name="vina")
v.set_receptor("protein_structures/chain_A_clean.pdbqt")
v.set_ligand_from_file("pdbqt/pyrimethamine_Br_optimized.pdbqt")
v.compute_vina_maps(
    center=pocket_center,
    box_size=[20, 20, 20]
)


mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1        -6.88          0          0
   2       -6.752       3.95       6.01
   3       -6.582       4.41      6.575
   4       -6.088      3.792      5.321
   5       -4.481      4.283      6.086
Computing Vina grid ... done.


In [39]:
os.makedirs("docking results", exist_ok=True)
v.dock(exhaustiveness=5, n_poses=5)
print("Docking results:")
for i, e in enumerate(v.energies(), 1):
    print(f"Pose {i}: {e[0]:.2f} kcal/mol")
v.write_poses(
    "docking results/pyrimethamine_poses.pdbqt",
    n_poses=5,
    overwrite=True
)

Performing docking (random seed: 1880480467) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Docking results:
Pose 1: -6.87 kcal/mol
Pose 2: -6.70 kcal/mol
Pose 3: -6.55 kcal/mol
Pose 4: -6.05 kcal/mol
Pose 5: -5.45 kcal/mol


In [48]:
from vina import Vina
from pathlib import Path
import os
ligand_dir = Path("pdbqt")
out_dir = Path("docking results")
out_dir.mkdir(exist_ok=True)
for ligand in ligand_dir.glob("*.pdbqt"):
    v = Vina(sf_name="vina")
    v.set_receptor("protein_structures/chain_A_clean.pdbqt")
    v.set_ligand_from_file(str(ligand))
    v.compute_vina_maps(
        center=pocket_center,
        box_size=[20, 20, 20]
    )
    v.dock(exhaustiveness=5, n_poses=5)
    print(f"\n{ligand.name}")
    for i, e in enumerate(v.energies(), 1):
        print(f"Pose {i}: {e[0]:.2f} kcal/mol")
    v.write_poses(
        str(out_dir / f"{ligand.stem}_poses.pdbqt"),
        n_poses=5,
        overwrite=True
    )


mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -6.861          0          0
   2       -6.744      3.931      6.001
   3       -5.446      3.729      4.962
Computing Vina grid ... 
pyrimethamine.pdbqt
Pose 1: -6.79 kcal/mol
Pose 2: -6.63 kcal/mol
Pose 3: -6.56 kcal/mol
done.
Performing docking (random seed: -1746162895) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -6.787          0          0
   2       -6.633      3.816      5.908
   3       -6.561       4.07      6.205
Computing Vina grid ... 
pyrimethamine_Br_optimized.pdbqt
Pose 1: -6.87 kcal/mol
Pose 2: -6.73 kcal/mol
Pose 3: -6.56 kcal/mol
Pose 4: -6.09 kcal/mol
Pose 5: -5.73 kcal/mol
done.
Perfo